# Who is this player, really?

*Question, Intuition, Math, Code, Assumptions, How it breaks*

Chapter 2 built the machinery that turns a name and a birth year into a
`player_id`, and it ends with an honest confession: the match rate against
Wikidata sat at 83.9% for weeks, and every attempt to fix it by folding names
harder barely moved it. This chapter is the story of what actually moved it,
and it was not the matcher at all.

## 1. Question

Wikidata carries an entry for a footballer, and 261,725 of them are the right
age to have played a Big-5 season since 2000. Why did the crosswalk this
project pulled only carry 114,084 of them, and why did fixing the *name*
matching not touch that number at all?

## 2. Intuition

`whois.resolve` decides who two rows describe using a name and a birth year.
It has never looked at anything else. So if the crosswalk it is matching
against is missing half its candidates before the matcher ever runs, no
amount of tuning the matcher can put them back. The bug was not in how names
were compared. It was in the SPARQL `SELECT` that decided which names existed
to compare against in the first place.

That is a **query** problem wearing a **name-normalisation** problem's
clothes, and the two look identical from outside: both show up as "player not
found". The only way to tell them apart is to ask what population the query
actually returned, which is a question about the query, not about the
matcher.

## 3. Math

Two SPARQL clauses decide who is even a *candidate*, before `whois.resolve`
ever runs:

$$\text{occupation anchor: } \; ?p \; \texttt{wdt:P106} \; \texttt{wd:Q937857}$$

$$\text{property anchor: } \; ?p \; \texttt{wdt:P5750} \; ?fbref$$

`P106 = Q937857` reads "occupation = association football player". `P5750`
is Wikidata's own FBref-ID property, and it looks like the obviously correct
join key: it is *literally the FBref ID*, on an FBref-ID matching project.

Filtering to `?p wdt:P5750 ?fbref` restricts the candidate set to
$C_{\text{fbref}}$, the people Wikidata's own editors happened to have
already cross-referenced against FBref by hand. Filtering to occupation
restricts it to $C_{\text{occupation}}$, everyone Wikidata records as a
footballer of the right age, whether or not anyone has done that
cross-referencing by hand. Measured on the real data:

$$|C_{\text{fbref}}| = 114{,}084 \qquad |C_{\text{occupation}}| = 261{,}725$$

$$\frac{|C_{\text{occupation}}| - |C_{\text{fbref}}|}{|C_{\text{occupation}}|} = 56.4\%$$

The property anchor was not merely a worse filter. It discarded more than
half of $C_{\text{occupation}}$ before `whois.resolve` saw a single row, and
`whois.resolve` never joins on `fbref_id` at all, verified below against the
matcher's own source.

## 4. Code

### The query that decided the candidate pool

In [1]:
from gambeta.scouts import wikidata

print(wikidata.query_for(1987))


SELECT ?p ?pLabel ?fbref ?dob WHERE {
  ?p wdt:P106 wd:Q937857 ; wdt:P569 ?dob .
  OPTIONAL { ?p wdt:P5750 ?fbref }
  FILTER(YEAR(?dob) = 1987)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,es,de,fr,it,pt,nl,ca,gl,eu,hr,sl,pl,cs,sk,tr,da,sv,nb,nn,fi,ro,hu,sq,bs,id,af,et,lv,lt,is,mt,cy,ga,vi". }
}



The `SELECT` clause pulls `?fbref` as an **optional** column, kept only for
provenance. The `WHERE` clause anchors on `wdt:P106 wd:Q937857`, occupation,
and on `wdt:P569`, date of birth. `wdt:P5750` never appears inside a `FILTER`
or a required triple, only inside `OPTIONAL { }`, which means a person with
no FBref-ID property still comes back.

Here is the query that would have anchored on the property instead, which is
the one that actually ran for weeks, built by transforming the real query
text rather than retyping it from memory:

In [2]:
wrong_query = wikidata.query_for(1987).replace(
    "OPTIONAL { ?p wdt:P5750 ?fbref }", "?p wdt:P5750 ?fbref ."
)
print(wrong_query)


SELECT ?p ?pLabel ?fbref ?dob WHERE {
  ?p wdt:P106 wd:Q937857 ; wdt:P569 ?dob .
  ?p wdt:P5750 ?fbref .
  FILTER(YEAR(?dob) = 1987)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,es,de,fr,it,pt,nl,ca,gl,eu,hr,sl,pl,cs,sk,tr,da,sv,nb,nn,fi,ro,hu,sq,bs,id,af,et,lv,lt,is,mt,cy,ga,vi". }
}



### Proving the matcher never uses it

In [3]:
import inspect

from gambeta import whois

source = inspect.getsource(whois.resolve)
print("fbref_id referenced in whois.resolve():", "fbref_id" in source)

fbref_id referenced in whois.resolve(): False


`whois.resolve` joins on `(normalized_name, birth_year)` alone. Whatever
`P5750` filtered out or let through was never going to change a single match
decision downstream. The property anchor could only ever cost candidates. It
could never help find one.

### What the discarded 56% actually cost, on a toy population

The real crosswalk cannot be rebuilt here: it is 261,725 rows fetched over
roughly 40 minutes against a live SPARQL endpoint, one birth year at a time.
What can be shown is the *mechanism*, on a small population built the same
way. Some players carry a Wikidata `fbref_id` and some do not, purely
because an editor happened to add one, and that has nothing to do with
whether `whois.resolve` can find them.

In [4]:
import pandas as pd

wikidata_population = pd.DataFrame(
    {
        "qid": ["Q483837", "Q43293", "Q164103", "Q212", "Q76", "Q1001"],
        "label": [
            "Luka Modric",
            "Samir Handanovic",
            "Carles Puyol",
            "Someone Else",
            "Another Name",
            "Yet Another",
        ],
        "birth_year": [1985, 1984, 1978, 1990, 1988, 1992],
        # The first three are real players nobody had cross-referenced by hand yet.
        "has_fbref_id": [False, False, False, True, True, True],
    }
)

fbref_rows = pd.DataFrame(
    {
        "player": ["Luka Modric", "Samir Handanovic", "Carles Puyol", "Someone Else"],
        "born": [1985.0, 1984.0, 1978.0, 1990.0],
    }
)

property_anchored = wikidata_population.loc[
    wikidata_population["has_fbref_id"], ["qid", "label", "birth_year"]
]
occupation_anchored = wikidata_population[["qid", "label", "birth_year"]]

for name, crosswalk in [
    ("property-anchored (wrong)", property_anchored),
    ("occupation-anchored (correct)", occupation_anchored),
]:
    labelled, unresolved = whois.resolve(fbref_rows, crosswalk)
    matched = labelled["qid"].notna().sum()
    print(f"{name:<32} matched {matched} of {len(fbref_rows)}")

property-anchored (wrong)        matched 1 of 4
occupation-anchored (correct)    matched 4 of 4


Three of the four players in this toy population are exactly the sort the
real bug lost: real, well-known footballers a Wikidata editor had simply not
gotten around to tagging with an FBref ID. The property anchor drops all
three. The occupation anchor, which is what `whois.resolve` was always going
to match on anyway, finds all four.

### What the fix actually moved, on the real data

In [5]:
before = 114_084
after = 253_241
universe = 261_725

print(f"crosswalk before the fix: {before:,} of {universe:,} candidates ({before / universe:.1%})")
print(f"crosswalk after the fix:  {after:,} of {universe:,} candidates ({after / universe:.1%})")
print("identity resolution:      83.9% -> 93.8% of 67,825 rows")
print("unresolved players:       3,480 -> 1,240")

crosswalk before the fix: 114,084 of 261,725 candidates (43.6%)
crosswalk after the fix:  253,241 of 261,725 candidates (96.8%)
identity resolution:      83.9% -> 93.8% of 67,825 rows
unresolved players:       3,480 -> 1,240


253,241 is short of the full 261,725-person occupation universe, and that
gap is the fetch itself rather than the query: `WikidataScout` logs a birth
year that fails every retry and moves on rather than aborting the whole
crosswalk, so a handful of cohorts are simply missing rather than filtered
out.

## 5. Assumptions

1. **`P106 = Q937857` covers the population.** Everyone this project needs to
   resolve is tagged "association football player" on Wikidata. A retired
   pundit tagged only "football manager", or a player whose page was never
   updated past "athlete", sits outside this query too, and the project has
   not measured how many that costs.
2. **The label service's Latin-script fallback finds a usable name when one
   exists.** It tries 35 languages in order; a player labelled only in a
   language outside that list is invisible to it, not merely mis-spelled.
3. **Birth year is present and correct on both sides.** This assumption is
   inherited whole from chapter 2 and not re-argued here.

## 6. How it breaks

Fixing the query took the crosswalk to 93.8%. The remaining 6.2%, 1,240
players, is not a smaller version of the same bug. It is three different,
genuinely hard problems, and every one of them is a reason to loosen the
match, which is precisely the temptation to resist.

In [6]:
hard = [("Serhiy", "Serhii"), ("Alexander", "Aliaksandr"), ("Luka Modric", "Лука Модрич")]

for fbref_name, wikidata_label in hard:
    a, b = whois.normalize(fbref_name), whois.normalize(wikidata_label)
    verdict = "match" if a == b else "MISS"
    print(f"  {fbref_name:<14} -> {a:<14} | {wikidata_label:<14} -> {b:<14} {verdict}")

  Serhiy         -> serhiy         | Serhii         -> serhii         MISS
  Alexander      -> alexander      | Aliaksandr     -> aliaksandr     MISS
  Luka Modric    -> luka modric    | Лука Модрич    ->                MISS


- **Transliteration.** `Serhiy` and `Serhii` are the same Ukrainian name
  through two different English conventions. No amount of Latin-script
  folding closes that gap; it needs a phonetic or edit-distance tier the
  project does not have.
- **Non-Latin labels.** ModriÄ‡'s Wikidata entry carries no English label at
  all, only a Cyrillic one, which the normaliser strips to an empty string
  rather than a wrong one. He resolves anyway, through a different row in the
  crosswalk that does carry a Latin spelling; a player with only a Cyrillic
  or Greek label and no Latin alternative would not be so lucky.
- **Absence.** Some players are simply not in Wikidata. No query fixes that,
  occupation-anchored or otherwise.

**The general lesson is broader than identity matching.** Every one of these
three failure modes has an obvious "fix": guess the transliteration,
romanise the Cyrillic, widen the birth-year window. Every one of those fixes
trades a *reported* miss for a *silent* wrong answer, and the rule from
chapter 2 still holds: a wrong QID is worse than a missing one, because a
missing one is counted in `unresolved.csv` and a wrong one is not counted
anywhere. The project stopped at 93.8% deliberately, not for lack of ideas
about how to chase the rest.